In [1]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, train_test_split as tts
import warnings 
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('/kaggle/input/iris/Iris.csv')
df.drop('Id',axis=1,inplace=True)
df.head()

,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


In [3]:
df.Species.value_counts()

Iris-setosa        50
Iris-versicolor    50
Iris-virginica     50
Name: Species, dtype: int64

In [4]:
df.replace({'Iris-setosa':0,'Iris-versicolor':1,'Iris-virginica':2},inplace=True)
df.head()

,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


In [5]:
x,y = df.iloc[:,:-1],df.iloc[:,-1]
x_train,x_test,y_train,y_test = tts(x,y,test_size=0.2)

In [6]:
from sklearn.base import BaseEstimator
class ClfSwitcher(BaseEstimator):
    def __init__(self, estimator = LogisticRegression()):
        """
        A Custom BaseEstimator that can switch between classifiers.
        :param estimator: sklearn object - The classifier
        """ 
        self.estimator = estimator

    def fit(self, X, y=None, **kwargs):
        self.estimator.fit(X, y)
        return self

    def predict(self, X, y=None):
        return self.estimator.predict(X)

    def predict_proba(self, X):
        return self.estimator.predict_proba(X)

    def score(self, X, y):
        return self.estimator.score(X, y)

In [7]:
pipeline = Pipeline([('clf', ClfSwitcher())])

parameters = [
    {
        'clf__estimator': [LogisticRegression()],
        'clf__estimator__max_iter': [50,80,100],
    },
    {
        'clf__estimator': [RandomForestClassifier()],
        'clf__estimator__n_estimators': [100,250,500],
        'clf__estimator__max_depth': [3,5,7],
        'clf__estimator__min_samples_split': [2,3]
    },
]

In [8]:
gscv = GridSearchCV(pipeline, parameters, cv=3, n_jobs=-1, verbose=2)
gscv.fit(x_train,y_train)

Fitting 3 folds for each of 21 candidates, totalling 63 fits


GridSearchCV(cv=3, estimator=Pipeline(steps=[('clf', ClfSwitcher())]),
             n_jobs=-1,
             param_grid=[{'clf__estimator': [LogisticRegression(max_iter=80)],
                          'clf__estimator__max_iter': [50, 80, 100]},
                         {'clf__estimator': [RandomForestClassifier()],
                          'clf__estimator__max_depth': [3, 5, 7],
                          'clf__estimator__min_samples_split': [2, 3],
                          'clf__estimator__n_estimators': [100, 250, 500]}],
             verbose=2)

In [9]:
gscv.best_params_

{'clf__estimator': LogisticRegression(max_iter=80),
 'clf__estimator__max_iter': 80}

In [10]:
gscv.best_estimator_

Pipeline(steps=[('clf',
                 ClfSwitcher(estimator=LogisticRegression(max_iter=80)))])